# Temporal held-out — pre-guideline / post-guideline

**Purpose.** Test whether prediction quality tracks the practice shift
documented in the JBI companion paper. Two variants:

- **`pre_to_post`**: train on admissions from August 2022 – July 2023,
  evaluate on August 2024 – December 2025.
- **`post_to_pre`**: train on post-guideline, evaluate on pre-guideline.

The intervening year (August 2023 – July 2024) is excluded from both
training and test — matching the JBI paper's rolling-implementation gap.

The pipeline supports this natively through the shared cohort's
`date_range_start` and `date_range_end` (applied to every source table
in one place). This notebook drives that switch, retrains the ladder,
and reads the results back into a comparison table.

**Compute.** A full ladder retrain per variant is required. Budget:
roughly the same as one canonical `run_ladder` execution.

In [ ]:
import os
import pandas as pd

from kineret.config import paths
from kineret.config import data_config as C
from kineret.config.data_config import configure_study
from kineret.benchmark import run_ladder, ensure_prepared, load_results

GUIDELINE_DATE  = "2023-09-01"   # National position paper published Sep 2023.
GAP_END         = "2024-08-01"   # One-year implementation gap after issue.
COHORT_START    = C.DATE_RANGE_START
COHORT_END      = C.DATE_RANGE_END

VARIANTS = {
    "pre_to_post":  dict(
        train=dict(date_range_start=COHORT_START, date_range_end=GUIDELINE_DATE),
        test =dict(date_range_start=GAP_END,      date_range_end=COHORT_END),
    ),
    "post_to_pre":  dict(
        train=dict(date_range_start=GAP_END,      date_range_end=COHORT_END),
        test =dict(date_range_start=COHORT_START, date_range_end=GUIDELINE_DATE),
    ),
}

## 2 — Runner

For each variant, retrain the ladder on the train-period cohort and
score on the held-out test-period cohort. The trained arms and their
predictions land under `outputs/temporal_holdout/<variant>/`.

This cell is a re-training driver — expect it to run for the same
wall-clock time as the canonical ladder. If you only want the summary
of an already-run experiment, skip to section 3.

In [ ]:
OUT_ROOT = paths.OUTPUT_ROOT

for variant, spec in VARIANTS.items():
    train_out = os.path.join(OUT_ROOT, "temporal_holdout", variant, "train")
    test_out  = os.path.join(OUT_ROOT, "temporal_holdout", variant, "test")
    print(f"\n=== {variant} ===")

    # 1. Train + in-period held-out on the training window.
    configure_study(**spec["train"])
    ensure_prepared(rebuild=True)
    run_ladder(output_root=train_out, skip_existing=True)

    # 2. Score on the disjoint test-period cohort.
    # This is where the current pipeline stops being a one-line switch:
    # `configure_study` re-builds a cohort limited to the test period,
    # and the trained models are loaded and re-scored against it.
    #
    # Load-and-score glue lives per-arm; the practical route is to run
    # each arm's `run(..., resume=True)` with the test-period cohort so
    # its held-out inference pass writes predictions for the target
    # period. That is left as the driver below to keep this notebook
    # explicit rather than magical.
    configure_study(**spec["test"])
    ensure_prepared(rebuild=True)
    # Deliberately not re-running training: `resume=True` reads the
    # previous run's checkpoints and produces predictions for this cohort.
    run_ladder(output_root=test_out, skip_existing=False,
               inference_only=True)   # <- flag currently a TODO in
                                      #    kineret/benchmark.run_ladder;
                                      #    see the note below.

# Restore the canonical config.
configure_study(date_range_start=COHORT_START, date_range_end=COHORT_END)

> **Note — `inference_only`.** The runner above assumes `run_ladder`
> accepts an `inference_only` flag; the current implementation always
> re-trains. A minimal patch is to thread `inference_only=True` through
> each arm's `run(..., resume=True, refresh_predictions_only=True)`;
> the arms already resume from the trained checkpoints, they just also
> take one gradient step or one epoch on the current cohort. Either
> land that flag before running this notebook, or run the second
> `run_ladder` without it and accept that it will start from the
> checkpoint but continue training on the test-period cohort — which
> would defeat the purpose of a held-out test.

A short PR to add the flag is on the AIIM task list.

## 3 — Summary

Compare each variant's held-out AUROC / AUPRC against the canonical
random-split numbers under `outputs/`.

In [ ]:
rows = []
for variant in VARIANTS:
    test_out = os.path.join(OUT_ROOT, "temporal_holdout", variant, "test")
    if not os.path.isdir(test_out):
        continue
    res = load_results(output_root=test_out, average="weighted")
    res["variant"] = variant
    rows.append(res)

canonical = load_results(average="weighted").assign(variant="random_split")
rows.append(canonical)

summary = pd.concat(rows, ignore_index=True)
summary.to_csv(os.path.join(OUT_ROOT, "temporal_holdout",
                             "summary.csv"), index=False)
summary.head(20)